# Independence Checker

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Contrast a 2×2 joint PMF with the independence product P(X)P(Y).

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/independence/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's defaults: P(0,0) = 0.4, P(0,1) = 0.1, P(1,0) = 0.15, P(1,1) = the rest
p00, p01, p10 = 0.4, 0.1, 0.15
P = torch.tensor([[p00, p01],
                  [p10, 1 - p00 - p01 - p10]], dtype=torch.float64)


def check(P, label, tol=1e-9):
    pX, pY = P.sum(1), P.sum(0)
    prod = torch.outer(pX, pY)
    tv = 0.5 * (P - prod).abs().sum().item()
    mi = (P * (P / prod).log()).sum().item()              # mutual information in nats
    print(f"{label}: TV(joint, P(X)P(Y)) = {tv:.4f}, mutual information = {mi:.4f} nats ->",
          "independent" if tv < tol else "dependent")
    return prod


prod = check(P, "default table")
print("P(X)P(Y) =", prod.tolist())
print("P(Y=1 | X=0) =", (P[0, 1] / P[0].sum()).item(), " vs P(Y=1 | X=1) =", (P[1, 1] / P[1].sum()).item())

# ── The demo's presets: independent means every cell is P(X=i) P(Y=j) ──────
for name, (a, b, c) in {"independent preset": (0.3, 0.2, 0.3), "dependent preset": (0.5, 0.05, 0.05)}.items():
    check(torch.tensor([[a, b], [c, 1 - a - b - c]], dtype=torch.float64), name)

# ── Independence from samples: estimate the table from draws ────────────────
n = 100000
x = (torch.rand(n) < 0.5).long()
y = (torch.rand(n) < 0.3).long()                          # drawn independently of x
counts = torch.zeros(2, 2, dtype=torch.float64)
counts.index_put_((x, y), torch.ones(n, dtype=torch.float64), accumulate=True)
# An estimated table is never exactly a product; judge it against sampling noise
check(counts / n, "\nindependent samples (estimated)", tol=0.01)

default table: TV(joint, P(X)P(Y)) = 0.2500, mutual information = 0.1325 nats -> dependent
P(X)P(Y) = [[0.275, 0.22499999999999998], [0.275, 0.22499999999999998]]
P(Y=1 | X=0) = 0.2  vs P(Y=1 | X=1) = 0.7
independent preset: TV(joint, P(X)P(Y)) = 0.0000, mutual information = 0.0000 nats -> independent
dependent preset: TV(joint, P(X)P(Y)) = 0.3950, mutual information = 0.3636 nats -> dependent

independent samples (estimated): TV(joint, P(X)P(Y)) = 0.0030, mutual information = 0.0000 nats -> independent



tensor([[0.3511, 0.1498],
        [0.3498, 0.1493]], dtype=torch.float64)